# Système de Recommandation TripAdvisor — Baseline BM25



## Pipeline
1. Chargement & Nettoyage des données
2. Stratégie de normalisation du nombre d'avis par lieu
3. Split Train / Test (50/50 par lieu)
4. Construction des représentations textuelles (indexation)
5. Baseline BM25
6. Évaluation — Niveau 1 (typeR) & Niveau 2 (catégories fines)
7. Interface pour brancher un modèle alternatif

---
## 0. Installation des dépendances

In [1]:
%pip install rank-bm25 nltk langdetect tqdm --quiet scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -r requirements.txt
%pip install pandas numpy --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 1. Imports & Configuration globale

In [3]:
import pandas as pd  
import numpy as np
import re
import ast
from collections import defaultdict
from typing import List, Dict, Tuple, Optional, Callable

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from tqdm import tqdm
import os
from pathlib import Path

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# ─── Configuration ───────────────────────────────────────────────────────────
RANDOM_SEED       = 42
# absolute data path (hardcoded since search failed)
data_dir = Path(r"C:\Users\axell\OneDrive\Bureau\Sup\A4\S2\IR&NLP\IR-NPL_Project1")
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory not found at {data_dir}")

REVIEWS_FILE = str(data_dir / 'reviews83325.csv')
PLACES_FILE  = str(data_dir / 'Tripadvisor.csv')

LANG_FILTER       = 'en'          # Garder uniquement les avis en anglais
MAX_REVIEWS_PLACE = 50            # Stratégie cap : max avis par lieu
TOP_TFIDF_WORDS   = 100           # Stratégie TF-IDF : top mots par lieu
STRATEGY          = 'cap'         # 'cap' | 'tfidf'  ← changez ici

np.random.seed(RANDOM_SEED)
print('Configuration chargée')
print('Using review file:', REVIEWS_FILE)
print('Using places file :', PLACES_FILE)

Configuration chargée
Using review file: C:\Users\axell\OneDrive\Bureau\Sup\A4\S2\IR&NLP\IR-NPL_Project1\reviews83325.csv
Using places file : C:\Users\axell\OneDrive\Bureau\Sup\A4\S2\IR&NLP\IR-NPL_Project1\Tripadvisor.csv


---
## 2. Chargement & Nettoyage des données

In [4]:
# ─── 2.1 Chargement ──────────────────────────────────────────────────────────
reviews_raw = pd.read_csv(REVIEWS_FILE, low_memory=False)
places_raw  = pd.read_csv(PLACES_FILE,  low_memory=False)

print(f'Reviews  : {reviews_raw.shape[0]:,} lignes  | colonnes : {list(reviews_raw.columns)}')
print(f'Places   : {places_raw.shape[0]:,} lignes  | colonnes : {list(places_raw.columns)}')

Reviews  : 340,385 lignes  | colonnes : ['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title']
Places   : 3,761 lignes  | colonnes : ['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbSc

In [ ]:
# ─── 2.2 Filtrage langue & nettoyage reviews ─────────────────────────────────
reviews = reviews_raw.copy()

reviews = reviews[reviews['langue'].str.lower() == LANG_FILTER].copy()
print(f'Avis EN uniquement : {len(reviews):,}')

if 'text' not in reviews.columns:
    candidates = [c for c in reviews.columns if c.lower().startswith('text') or c.lower().startswith('review')]
    if candidates:
        print(f" '{candidates[0]}' '")
        reviews = reviews.rename(columns={candidates[0]: 'text'})
    else:
        raise KeyError(f"Colonne 'text' introuvable dans reviews, colonnes disponibles : {reviews.columns.tolist()}")

reviews = reviews.dropna(subset=['text']).copy()
reviews['text'] = reviews['text'].astype(str).str.strip()
reviews = reviews[reviews['text'].str.len() > 10].copy()

reviews['idplace'] = reviews['idplace'].astype(str).str.strip()

print(f'Après nettoyage : {len(reviews):,} avis')

Avis EN uniquement : 153,071
 'review' '
Après nettoyage : 153,066 avis


In [ ]:
# ─── 2.3 Nettoyage places & normalisation clé ────────────────────────────────
places = places_raw.copy()
places['id'] = places['id'].astype(str).str.strip()

# Parser les colonnes liste
def safe_parse_list(val):
    if pd.isna(val) or val == '' or val == 'nan':
        return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [str(val)]

LIST_COLS = ['activiteSubCategorie', 'activiteSubType', 'restaurantType', 'cuisine']
for col in LIST_COLS:
    if col in places.columns:
        places[col] = places[col].apply(safe_parse_list)

print(f'Places nettoyées : {len(places):,}')
places[['id', 'typeR'] + [c for c in LIST_COLS if c in places.columns]].head(10)

Places nettoyées : 3,761


,id,typeR,activiteSubCategorie,activiteSubType,restaurantType
0,188467,A,[47],[163],[]
1,188468,A,[47],[163],[]
2,188470,A,"[(26, 47, 51)]","[(34, 144)]",[]
3,188471,A,[26],[137],[]
4,188472,A,[47],[10],[]
5,188493,A,[26],[144],[]
6,188679,A,[47],"[(3, 175, 17)]",[]
7,188738,H,[],[],[]
8,188745,H,[],[],[]
9,188758,A,[49],[161],[]


In [ ]:
# ─── 2.4 Jointure ────────────────────────────────────────────────────────────
# On joint uniquement sur les places qui ont au moins un avis
required_cols = ['id', 'typeR']
optional_cols = ['activiteSubCategorie', 'activiteSubType',
                 'restaurantType', 'cuisine', 'priceRange']
meta_cols = required_cols + [c for c in optional_cols if c in places.columns]

reviews_with_meta = reviews.merge(
    places[meta_cols],
    left_on='idplace', right_on='id',
    how='inner'
)

# Lieux valides = ceux ayant au moins un avis EN
valid_place_ids = reviews_with_meta['idplace'].unique()
places = places[places['id'].isin(valid_place_ids)].copy().reset_index(drop=True)

print(f'Avis jointure : {len(reviews_with_meta):,}')
print(f'Lieux valides : {len(places):,}')
print('Distribution typeR :', reviews_with_meta['typeR'].value_counts().to_dict())

Avis jointure : 153,066
Lieux valides : 1,835
Distribution typeR : {'A': 70059, 'R': 44507, 'AP': 24779, 'H': 13721}


---
## 3. Stratégie de normalisation du nombre d'avis

In [8]:
# ─── 3.1 Visualisation du déséquilibre ───────────────────────────────────────
review_counts = reviews_with_meta.groupby('idplace').size()
print('Stats avis/lieu :')
print(review_counts.describe().to_string())
print(f'\nLieux avec >100 avis : {(review_counts > 100).sum()}')
print(f'Lieux avec <5  avis  : {(review_counts < 5).sum()}')

Stats avis/lieu :
count     1835.000000
mean        83.414714
std        828.874642
min          1.000000
25%          2.000000
50%          8.000000
75%         36.000000
max      33878.000000

Lieux avec >100 avis : 244
Lieux avec <5  avis  : 763


In [ ]:
# ─── 3.2 Stratégie CAP — garder MAX_REVIEWS_PLACE avis par lieu ──────────────
def strategy_cap(df: pd.DataFrame, max_reviews: int, seed: int = 42) -> pd.DataFrame:
    
    return (
        df.sample(frac=1, random_state=seed)          
          .groupby('idplace')                         
          .head(max_reviews)                          
          .reset_index(drop=True)                    
    )


# ─── 3.3 Stratégie TF-IDF — top N mots représentatifs ────────────────────────
def strategy_tfidf_keywords(df: pd.DataFrame, top_n: int = 100) -> pd.DataFrame:
    
    place_docs = df.groupby('idplace')['text'].apply(lambda t: ' '.join(t)).reset_index()
    place_docs.columns = ['idplace', 'full_text']

    vectorizer = TfidfVectorizer(max_features=None, stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(place_docs['full_text'])
    feature_names = np.array(vectorizer.get_feature_names_out())

    keyword_docs = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray().flatten()
        top_idx = row.argsort()[::-1][:top_n]
        keywords = ' '.join(feature_names[top_idx])
        keyword_docs.append({'idplace': place_docs.iloc[i]['idplace'], 'text': keywords})

    return pd.DataFrame(keyword_docs)


# ─── 3.4 Application de la stratégie choisie ─────────────────────────────────
print('Colonnes reviews_with_meta avant normalisation :', reviews_with_meta.columns.tolist())
if 'idplace' not in reviews_with_meta.columns:
    raise KeyError(f"'idplace' introuvable dans reviews_with_meta, colonnes = {reviews_with_meta.columns.tolist()}")

if STRATEGY == 'cap':
    reviews_balanced = strategy_cap(reviews_with_meta, MAX_REVIEWS_PLACE, RANDOM_SEED)
    # Concaténer les avis par lieu pour former un document
    place_documents = (
        reviews_balanced
        .groupby('idplace')['text']
        .apply(lambda t: ' '.join(t.astype(str)))
        .reset_index()
        .rename(columns={'text': 'document'})
    )
elif STRATEGY == 'tfidf':
    tfidf_df = strategy_tfidf_keywords(reviews_with_meta, TOP_TFIDF_WORDS)
    place_documents = tfidf_df.rename(columns={'text': 'document'})
else:
    raise ValueError(f'Stratégie inconnue : {STRATEGY}')

# Joindre les métadonnées
place_documents = place_documents.merge(places, left_on='idplace', right_on='id', how='inner')
print(f'Documents construits : {len(place_documents):,} lieux')
place_documents[['idplace', 'typeR', 'document']].head(10)

Colonnes reviews_with_meta avant normalisation : ['id_x', 'idplace', 'titre', 'idauteur', 'text', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title', 'id_y', 'typeR', 'activiteSubCategorie', 'activiteSubType', 'restaurantType', 'priceRange']
Documents construits : 1,835 lieux


,idplace,typeR,document
0,10041740,R,"Superb food, warm and gracious service, lovely..."
1,10055668,R,"Nice staff, wonderful advices, and incredible ..."
2,10089740,A,My wife and I decided to splurge and book a fa...
3,10105969,A,This is a small art gallery on Place des Vosge...
4,10131114,R,This oas a cute place for teatime. The cakes ...
5,1013459,R,This place is famous and there is no doubt as ...
6,10163512,A,After visiting the Island of Saint Louis and e...
7,10163562,A,This was another bridge that linked to the sma...
8,10171785,A,"A good looking wooden bridge, offering fantast..."
9,10180824,A,Sit down and rest awhile behind Notre Dame and...


---
## 4. Split Train / Test (50 / 50 par lieu)

In [11]:
from sklearn.model_selection import train_test_split

all_place_ids = place_documents['idplace'].unique()

train_ids, test_ids = train_test_split(
    all_place_ids,
    test_size=0.5,
    random_state=RANDOM_SEED
)

train_docs = place_documents[place_documents['idplace'].isin(train_ids)].reset_index(drop=True)
test_docs  = place_documents[place_documents['idplace'].isin(test_ids)].reset_index(drop=True)

print(f'Train : {len(train_docs):,} lieux')
print(f'Test  : {len(test_docs):,} lieux')
print('\nDistribution typeR — Train :', train_docs['typeR'].value_counts().to_dict())
print('Distribution typeR — Test  :', test_docs['typeR'].value_counts().to_dict())

Train : 917 lieux
Test  : 918 lieux

Distribution typeR — Train : {'AP': 494, 'R': 271, 'A': 118, 'H': 34}
Distribution typeR — Test  : {'AP': 495, 'R': 267, 'A': 134, 'H': 22}


---
## 5. Prétraitement NLP & Tokenisation

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
stemmer    = PorterStemmer()

def preprocess_text(text: str, stem: bool = True) -> List[str]:
    
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# Tokenisation des documents (corpus de test = index de recherche)
print('Tokenisation du corpus de test...')
test_docs['tokens'] = [preprocess_text(doc) for doc in tqdm(test_docs['document'])]

print('Tokenisation du corpus de train (requêtes)...')
train_docs['tokens'] = [preprocess_text(doc) for doc in tqdm(train_docs['document'])]

print(' Tokenisation terminée')

Tokenisation du corpus de test...


  0%|          | 0/918 [00:00<?, ?it/s]

100%|██████████| 918/918 [00:25<00:00, 35.73it/s]


Tokenisation du corpus de train (requêtes)...


100%|██████████| 917/917 [00:22<00:00, 40.09it/s] 

 Tokenisation terminée


---
## 6. Baseline BM25

> **Architecture** : le modèle est indexé sur les lieux du **set de test** et interrogé par les lieux du **set de train** (chaque lieu train = une requête).

In [ ]:
# ─── 6.1 Interface générique du modèle de recherche ──────────────────────────
class RetrievalModel:
    
    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        raise NotImplementedError

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        
        raise NotImplementedError


# ─── 6.2 Implémentation BM25 ──────────────────────────────────────────────────
class BM25Model(RetrievalModel):
    """
    Baseline BM25 (Okapi) via rank-bm25.
    k1=1.5, b=0.75 — paramètres standards TREC.
    """
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.bm25 = None

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        self.bm25 = BM25Okapi(corpus_tokens, k1=self.k1, b=self.b)
        print(f' Index BM25 construit — {len(corpus_tokens):,} documents')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        return self.bm25.get_scores(query_tokens)


# ─── 6.3 Construction de l'index ─────────────────────────────────────────────
bm25_model = BM25Model()
bm25_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

 Index BM25 construit — 918 documents


---
## 7. Fonctions d'évaluation

In [ ]:
# ─── 7.1 Helpers de correspondance de métadonnées ────────────────────────────

def match_level1(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    return query_row['typeR'] == candidate_row['typeR']


def match_level2(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    
    type_r = query_row.get('typeR', '')

    def lists_overlap(a, b) -> bool:
        set_a = set(a) if isinstance(a, list) else {a}
        set_b = set(b) if isinstance(b, list) else {b}
        return bool(set_a & set_b)

    if type_r in ('A', 'AP'):   
        sub_cat   = lists_overlap(query_row.get('activiteSubCategorie', []),
                                  candidate_row.get('activiteSubCategorie', []))
        sub_type  = lists_overlap(query_row.get('activiteSubType', []),
                                  candidate_row.get('activiteSubType', []))
        return sub_cat or sub_type

    elif type_r == 'R':         
        rest_type = lists_overlap(query_row.get('restaurantType', []),
                                  candidate_row.get('restaurantType', []))
        cuisine   = lists_overlap(query_row.get('cuisine', []),
                                  candidate_row.get('cuisine', []))
        return rest_type or cuisine

    elif type_r == 'H':         
        q_price = query_row.get('priceRange', None)
        c_price = candidate_row.get('priceRange', None)
        if pd.isna(q_price) or pd.isna(c_price) or q_price == '' or c_price == '':
            return False
        return q_price == c_price

    return False 


def has_valid_candidate(query_row: pd.Series,
                        test_df: pd.DataFrame,
                        match_fn: Callable) -> bool:
   
    return any(match_fn(query_row, test_df.iloc[i]) for i in range(len(test_df)))


print('Fonctions de correspondance définies')

Fonctions de correspondance définies


In [ ]:
# ─── 7.2 Calcul du Ranking Error ─────────────────────────────────────────────

def compute_ranking_error(query_row: pd.Series,
                          scores: np.ndarray,
                          test_df: pd.DataFrame,
                          match_fn: Callable) -> Optional[int]:
    
    ranked_indices = np.argsort(scores)[::-1]  

    for rank, idx in enumerate(ranked_indices):
        candidate = test_df.iloc[idx]
        if match_fn(query_row, candidate):
            return rank   

    return None  

def evaluate_model(model: RetrievalModel,
                   train_df: pd.DataFrame,
                   test_df: pd.DataFrame,
                   match_fn: Callable,
                   level_name: str = 'L1') -> Dict:
    
    errors    = []
    n_skipped = 0

    for _, query_row in tqdm(train_df.iterrows(), total=len(train_df),
                             desc=f'Évaluation {level_name}'):

        if not has_valid_candidate(query_row, test_df, match_fn):
            n_skipped += 1
            continue

        scores = model.rank(
            query_tokens=query_row['tokens'],
            query_text=query_row.get('document', '')
        )

        err = compute_ranking_error(query_row, scores, test_df, match_fn)
        if err is not None:
            errors.append(err)

    results = {
        'errors'       : errors,
        'mean_error'   : np.mean(errors)   if errors else float('nan'),
        'median_error' : np.median(errors) if errors else float('nan'),
        'success_at_1' : np.mean([e == 0 for e in errors]) * 100 if errors else 0.0,
        'n_queries'    : len(errors),
        'n_skipped'    : n_skipped,
    }
    return results


print('Fonctions d\'évaluation définies')

Fonctions d'évaluation définies


In [ ]:
def match_level2_v2(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    type_r = query_row.get('typeR', '')

    def lists_overlap(a, b) -> bool:
        set_a = set(a) if isinstance(a, list) else {a}
        set_b = set(b) if isinstance(b, list) else {b}
        return bool(set_a & set_b)

    if type_r in ('A', 'AP'):
        sub_cat   = lists_overlap(query_row.get('activiteSubCategorie', []),
                                  candidate_row.get('activiteSubCategorie', []))
        sub_type  = lists_overlap(query_row.get('activiteSubType', []),
                                  candidate_row.get('activiteSubType', []))
        return sub_cat or sub_type

    elif type_r == 'R':
        rest_type = lists_overlap(query_row.get('restaurantType', []),
                                  candidate_row.get('restaurantType', []))
        cuisine   = lists_overlap(query_row.get('cuisine', []),
                                  candidate_row.get('cuisine', []))
        return rest_type or cuisine

    elif type_r == 'H':
        q_price = query_row.get('priceRange', None)
        c_price = candidate_row.get('priceRange', None)
        
        if pd.notna(q_price) and pd.notna(c_price) and q_price != '' and c_price != '':
            return q_price == c_price
        
        return True

    return False

res_l2_v2 = evaluate_model(bm25_model, train_docs, test_docs, match_level2_v2, 'L2_v2')
print(f"L2 — Mean Error: {res_l2_v2['mean_error']:.2f} | Success@1: {res_l2_v2['success_at_1']:.1f}%")

Évaluation L2_v2: 100%|██████████| 917/917 [05:03<00:00,  3.02it/s]

L2 (assouplies) — Mean Error: 9.11 | Success@1: 71.5%


In [ ]:
def match_level2_ap_fallback(query_row: pd.Series, candidate_row: pd.Series) -> bool:
    
    type_r = query_row.get('typeR', '')

    def lists_overlap(a, b) -> bool:
        set_a = set(a) if isinstance(a, list) else {a}
        set_b = set(b) if isinstance(b, list) else {b}
        return bool(set_a & set_b)

    if type_r in ('A', 'AP'):
        sub_cat = lists_overlap(query_row.get('activiteSubCategorie', []),
                                candidate_row.get('activiteSubCategorie', []))
        sub_type = lists_overlap(query_row.get('activiteSubType', []),
                                 candidate_row.get('activiteSubType', []))
        
        #  Match si catégories trouvées
        if sub_cat or sub_type:
            return True
        
        #  Fallback à L1 (juste typeR) — pour AP/A sans métadonnées
        return query_row['typeR'] == candidate_row['typeR']

    elif type_r == 'R':
        rest_type = lists_overlap(query_row.get('restaurantType', []),
                                  candidate_row.get('restaurantType', []))
        cuisine = lists_overlap(query_row.get('cuisine', []),
                               candidate_row.get('cuisine', []))
        return rest_type or cuisine

    elif type_r == 'H':
        q_price = query_row.get('priceRange', None)
        c_price = candidate_row.get('priceRange', None)
        if pd.notna(q_price) and pd.notna(c_price) and q_price != '' and c_price != '':
            return q_price == c_price
        return True

    return False

# ─── Test ───────────────────────────────────────────────────────────────────
print('=' * 55)
print('ÉVALUATION NIVEAU 2 AP_FALLBACK — avec fallback AP→L1')
print('=' * 55)

res_l2_ap_fallback = evaluate_model(
    model      = bm25_model,
    train_df   = train_docs,
    test_df    = test_docs,
    match_fn   = match_level2_ap_fallback,
    level_name = 'L2_AP_fallback'
)

print(f"\n Résultats Niveau 2 AP_fallback :")
print(f"  Requêtes évaluées : {res_l2_ap_fallback['n_queries']:,}")
print(f"  Requêtes ignorées : {res_l2_ap_fallback['n_skipped']:,}")
print(f"  Ranking Error moyen   : {res_l2_ap_fallback['mean_error']:.2f}")
print(f"  Ranking Error médian  : {res_l2_ap_fallback['median_error']:.2f}")
print(f"  Success@1 (erreur=0)  : {res_l2_ap_fallback['success_at_1']:.1f}%")

ÉVALUATION NIVEAU 2 AP_FALLBACK — avec fallback AP→L1


Évaluation L2_AP_fallback: 100%|██████████| 917/917 [04:13<00:00,  3.62it/s]


 Résultats Niveau 2 AP_fallback :
  Requêtes évaluées : 914
  Requêtes ignorées : 3
  Ranking Error moyen   : 1.41
  Ranking Error médian  : 0.00
  Success@1 (erreur=0)  : 83.0%


---
## 8. Évaluation BM25 — Niveau 1 & Niveau 2

In [20]:
# ─── Niveau 1 : typeR ─────────────────────────────────────────────────────────
print('=' * 55)
print('ÉVALUATION NIVEAU 1 — typeR (H / R / A / AP)')
print('=' * 55)

results_l1 = evaluate_model(
    model      = bm25_model,
    train_df   = train_docs,
    test_df    = test_docs,
    match_fn   = match_level1,
    level_name = 'L1'
)

print(f"\n Résultats Niveau 1 :")
print(f"  Requêtes évaluées : {results_l1['n_queries']}")
print(f"  Requêtes ignorées : {results_l1['n_skipped']} (pas de candidat valide)")
print(f"  Ranking Error moyen   : {results_l1['mean_error']:.2f}")
print(f"  Ranking Error médian  : {results_l1['median_error']:.2f}")
print(f"  Success@1 (erreur=0)  : {results_l1['success_at_1']:.1f}%")

ÉVALUATION NIVEAU 1 — typeR (H / R / A / AP)


Évaluation L1: 100%|██████████| 917/917 [04:36<00:00,  3.31it/s]


 Résultats Niveau 1 :
  Requêtes évaluées : 917
  Requêtes ignorées : 0 (pas de candidat valide)
  Ranking Error moyen   : 0.63
  Ranking Error médian  : 0.00
  Success@1 (erreur=0)  : 88.4%


In [ ]:
# ─── Tableau récapitulatif ────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Modèle'         : ['BM25', 'BM25'],
    'Niveau'         : ['L1 — typeR', 'L2 AP fallback'],
    'Requêtes'       : [results_l1['n_queries'], res_l2_ap_fallback['n_queries']],
    'Mean Error ↓'   : [round(results_l1['mean_error'], 3), round(res_l2_ap_fallback['mean_error'], 3)],
    'Median Error ↓' : [round(results_l1['median_error'], 3), round(res_l2_ap_fallback['median_error'], 3)],
    'Success@1 ↑ (%)': [round(results_l1['success_at_1'], 1), round(res_l2_ap_fallback['success_at_1'], 1)],
})
print('\n', summary.to_string(index=False))


 Modèle           Niveau  Requêtes  Mean Error ↓  Median Error ↓  Success@1 ↑ (%)
  BM25       L1 — typeR       917         0.627             0.0             88.4
  BM25 L2 v2 — assoupli       410         9.107             0.0             71.5


---
## 9. Implementation des modèles : TF-IDF cosinus 

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class TFIDFCosineModel(RetrievalModel):
    
    def __init__(self):
        self.vectorizer    = TfidfVectorizer(analyzer='word', stop_words='english')
        self.corpus_matrix = None

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        # TF-IDF travaille sur du texte brut
        docs = corpus_texts if corpus_texts else [' '.join(t) for t in corpus_tokens]
        self.corpus_matrix = self.vectorizer.fit_transform(docs)
        print(f' Index TF-IDF construit — {self.corpus_matrix.shape[0]:,} documents')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        query_str    = query_text if query_text else ' '.join(query_tokens)
        query_vector = self.vectorizer.transform([query_str])
        scores       = cosine_similarity(query_vector, self.corpus_matrix).flatten()
        return scores


# ─── Construction et évaluation ───────────────────────────────────────────────
tfidf_model = TFIDFCosineModel()
tfidf_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

res_tfidf_l1 = evaluate_model(tfidf_model, train_docs, test_docs, match_level1, 'TF-IDF L1')
res_tfidf_l2 = evaluate_model(tfidf_model, train_docs, test_docs, match_level2, 'TF-IDF L2')

print(f"\nTF-IDF L1 — Mean Error: {res_tfidf_l1['mean_error']:.2f} | Success@1: {res_tfidf_l1['success_at_1']:.1f}%")
print(f"TF-IDF L2 — Mean Error: {res_tfidf_l2['mean_error']:.2f} | Success@1: {res_tfidf_l2['success_at_1']:.1f}%")

 Index TF-IDF construit — 918 documents


Évaluation TF-IDF L2: 100%|██████████| 917/917 [03:27<00:00,  4.41it/s]


TF-IDF L1 — Mean Error: 0.79 | Success@1: 88.7%
TF-IDF L2 — Mean Error: 11.07 | Success@1: 70.1%


In [29]:
# ─── Tableau de comparaison final ────────────────────────────────────────────
comparison = pd.DataFrame([
    {'Modèle': 'BM25',   'Niveau': 'L1', 'Mean Error': results_l1['mean_error'],    'Success@1 (%)': results_l1['success_at_1']},
    {'Modèle': 'BM25',   'Niveau': 'L2', 'Mean Error': res_l2_ap_fallback['mean_error'],    'Success@1 (%)': res_l2_ap_fallback['success_at_1']},
    {'Modèle': 'TF-IDF', 'Niveau': 'L1', 'Mean Error': res_tfidf_l1['mean_error'], 'Success@1 (%)': res_tfidf_l1['success_at_1']},
    {'Modèle': 'TF-IDF', 'Niveau': 'L2', 'Mean Error': res_tfidf_l2['mean_error'], 'Success@1 (%)': res_tfidf_l2['success_at_1']},
    {'Modèle': 'SBERT',  'Niveau': 'L1', 'Mean Error': res_sbert_l1['mean_error'],  'Success@1 (%)': res_sbert_l1['success_at_1']},
    {'Modèle': 'SBERT',  'Niveau': 'L2', 'Mean Error': res_sbert_l2['mean_error'],  'Success@1 (%)': res_sbert_l2['success_at_1']},
    {'Modèle': 'Hybrid', 'Niveau': 'L1', 'Mean Error': res_hybrid_l1['mean_error'], 'Success@1 (%)': res_hybrid_l1['success_at_1']},
    {'Modèle': 'Hybrid', 'Niveau': 'L2', 'Mean Error': res_hybrid_l2['mean_error'], 'Success@1 (%)': res_hybrid_l2['success_at_1']},
])

comparison['Mean Error'] = comparison['Mean Error'].round(3)
comparison['Success@1 (%)'] = comparison['Success@1 (%)'].round(1)
print('\n' + '═' * 70)
print(' COMPARAISON FINALE')
print('═' * 70)
print(comparison.to_string(index=False))


══════════════════════════════════════════════════════════════════════
 COMPARAISON FINALE
══════════════════════════════════════════════════════════════════════
Modèle Niveau  Mean Error  Success@1 (%)
  BM25     L1       0.627           88.4
  BM25     L2       1.410           83.0
TF-IDF     L1       0.795           88.7
TF-IDF     L2      11.069           70.1
 SBERT     L1       0.846           88.2
 SBERT     L2      12.218           67.4
Hybrid     L1       0.794           89.9
Hybrid     L2       9.429           70.8


---
## 9bis. SBERT (Sentence-BERT) — Embeddings sémantiques

In [26]:
# ─── Installation de sentence-transformers ──────────────────────────────────
%pip install sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class SBERTModel(RetrievalModel):
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.corpus_embeddings = None
        print(f"SBERT chargé : {model_name}")

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        # SBERT fonctionne sur du texte brut
        docs = corpus_texts if corpus_texts else [' '.join(t) for t in corpus_tokens]
        print(f"Encodage de {len(docs):,} documents SBERT...")
        self.corpus_embeddings = self.model.encode(docs, show_progress_bar=True, batch_size=32)
        print(f'Index SBERT construit — {len(docs):,} documents')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        query_str = query_text if query_text else ' '.join(query_tokens)
        query_emb = self.model.encode([query_str])
        scores = cosine_similarity(query_emb, self.corpus_embeddings).flatten()
        return scores


# ─── Construction et évaluation SBERT ────────────────────────────────────────
sbert_model = SBERTModel()
sbert_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

res_sbert_l1 = evaluate_model(sbert_model, train_docs, test_docs, match_level1, 'SBERT L1')
res_sbert_l2 = evaluate_model(sbert_model, train_docs, test_docs, match_level2, 'SBERT L2')

print(f"\nSBERT L1 — Mean Error: {res_sbert_l1['mean_error']:.2f} | Success@1: {res_sbert_l1['success_at_1']:.1f}%")
print(f"SBERT L2 — Mean Error: {res_sbert_l2['mean_error']:.2f} | Success@1: {res_sbert_l2['success_at_1']:.1f}%")

c:\Users\axell\OneDrive\Bureau\Sup\A4\S2\IR&NLP\IR-NPL_Project1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\axell\OneDrive\Bureau\Sup\A4\S2\IR&NLP\IR-NPL_Project1\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\axell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to ac

SBERT chargé : all-MiniLM-L6-v2
Encodage de 918 documents SBERT...


Batches: 100%|██████████| 29/29 [00:58<00:00,  2.01s/it]


Index SBERT construit — 918 documents


Évaluation SBERT L2: 100%|██████████| 917/917 [03:51<00:00,  3.96it/s]


SBERT L1 — Mean Error: 0.85 | Success@1: 88.2%
SBERT L2 — Mean Error: 12.22 | Success@1: 67.4%


---
## 9ter. Modèle Hybrid — Fusion BM25 + SBERT

In [ ]:
class HybridModel(RetrievalModel):
    
    def __init__(self, bm25_model: BM25Model, sbert_model: SBERTModel, alpha: float = 0.5):
        self.bm25_model = bm25_model
        self.sbert_model = sbert_model
        self.alpha = alpha  

    def build_index(self, corpus_tokens: List[List[str]], corpus_texts: List[str] = None):
        print(f'Index Hybrid configuré (α={self.alpha} BM25 + {1-self.alpha} SBERT)')

    def rank(self, query_tokens: List[str], query_text: str = None) -> np.ndarray:
        bm25_scores = self.bm25_model.rank(query_tokens, query_text)
        sbert_scores = self.sbert_model.rank(query_tokens, query_text)
        
        bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-10)
        sbert_norm = (sbert_scores - sbert_scores.min()) / (sbert_scores.max() - sbert_scores.min() + 1e-10)
        
        hybrid_scores = self.alpha * bm25_norm + (1 - self.alpha) * sbert_norm
        return hybrid_scores


# ─── Construction et évaluation Hybrid ───────────────────────────────────────
hybrid_model = HybridModel(bm25_model, sbert_model, alpha=0.5)
hybrid_model.build_index(
    corpus_tokens=test_docs['tokens'].tolist(),
    corpus_texts=test_docs['document'].tolist()
)

res_hybrid_l1 = evaluate_model(hybrid_model, train_docs, test_docs, match_level1, 'Hybrid L1')
res_hybrid_l2 = evaluate_model(hybrid_model, train_docs, test_docs, match_level2, 'Hybrid L2')

print(f"\nHybrid L1 — Mean Error: {res_hybrid_l1['mean_error']:.2f} | Success@1: {res_hybrid_l1['success_at_1']:.1f}%")
print(f"Hybrid L2 — Mean Error: {res_hybrid_l2['mean_error']:.2f} | Success@1: {res_hybrid_l2['success_at_1']:.1f}%")

Index Hybrid configuré (α=0.5 BM25 + 0.5 SBERT)


Évaluation Hybrid L1:   0%|          | 0/917 [00:00<?, ?it/s]

Évaluation Hybrid L2: 100%|██████████| 917/917 [05:19<00:00,  2.87it/s]


Hybrid L1 — Mean Error: 0.79 | Success@1: 89.9%
Hybrid L2 — Mean Error: 9.43 | Success@1: 70.8%
